In [ ]:
import os


In [34]:
!install -q youtube-transcript-api langchain-community langchain-openai\faiss-cpu tiktoken python-dotenv

install: invalid option -- 'q'
Try 'install --help' for more information.


In [35]:
from youtube_transcript_api import YouTubeTranscriptApi     
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate


In [36]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, VideoUnavailable

# Replace this with the actual 11-character ID after "v=" in your YouTube link
video_id = "FlqItH637no"

try:
    ytt_api = YouTubeTranscriptApi()
    fetched_transcript = ytt_api.fetch(video_id, languages=['en'])
    transcript = "\n".join(f"[{snippet.start:.1f}s] {snippet.text}" for snippet in fetched_transcript)
    print("Transcript fetched successfully.",transcript)

except TranscriptsDisabled:
    print("Transcript is disabled for this video.")
except VideoUnavailable:
    print("Video is unavailable or the video_id is invalid.")

Transcript fetched successfully. [5.5s] Hello and welcome back to Leo English
[8.2s] podcast. We're very happy you're here
[10.8s] with us today.
[11.9s] >> I'm Leo.
[12.8s] >> And I'm Tina. Today's topic is very
[15.9s] important. We are talking about how to
[18.4s] train your brain to speak in English.
[21.5s] >> And also how to think in English, not
[23.4s] only study English.
[25.0s] >> That's right. Many learners know grammar
[27.6s] and vocabulary, but when they want to
[29.8s] speak, they stop, translate, wait, and
[34.0s] feel nervous.
[35.2s] >> I know that feeling. Your brain says,
[37.4s] "Come on, say something." And your mouth
[39.5s] says, "No, thank you."
[41.2s] >> Exactly. So, today we will show you
[44.4s] practical ways to make English feel more
[46.9s] natural in your mind.
[48.7s] >> We'll talk about daily habits, simple
[50.6s] practice, common mistakes, and useful
[53.0s] phrases you can start using right away.
[55.6s] >> If you want more easy English lessons
[57

In [37]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks=splitter.create_documents([transcript])

In [38]:
len(chunks)

47

In [39]:
chunks[0]

Document(metadata={}, page_content='[5.5s] Hello and welcome back to Leo English\n[8.2s] podcast. We\'re very happy you\'re here\n[10.8s] with us today.\n[11.9s] >> I\'m Leo.\n[12.8s] >> And I\'m Tina. Today\'s topic is very\n[15.9s] important. We are talking about how to\n[18.4s] train your brain to speak in English.\n[21.5s] >> And also how to think in English, not\n[23.4s] only study English.\n[25.0s] >> That\'s right. Many learners know grammar\n[27.6s] and vocabulary, but when they want to\n[29.8s] speak, they stop, translate, wait, and\n[34.0s] feel nervous.\n[35.2s] >> I know that feeling. Your brain says,\n[37.4s] "Come on, say something." And your mouth\n[39.5s] says, "No, thank you."\n[41.2s] >> Exactly. So, today we will show you\n[44.4s] practical ways to make English feel more\n[46.9s] natural in your mind.\n[48.7s] >> We\'ll talk about daily habits, simple\n[50.6s] practice, common mistakes, and useful\n[53.0s] phrases you can start using right away.\n[55.6s] >> If you wa

In [40]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vector_store = FAISS.from_documents(chunks, embeddings)

In [41]:
vector_store.index_to_docstore_id

{0: '15e1907f-ace0-47e8-8828-00a623a180db',
 1: 'd0bd7f1d-7576-4656-8257-716092c30408',
 2: 'dd920f51-3dee-4189-a5fb-5603f8bac12d',
 3: '95b6ee4c-6724-4b33-85ce-2d7c68dc3471',
 4: '8f49159c-4298-4c86-b9fd-7190b814acee',
 5: '35d8faf9-a141-497a-b666-aceb0e5743a3',
 6: '62a390ab-5140-433c-ae01-95b420d37b3e',
 7: 'ad8ff796-5502-4d1f-a33d-5e7831a2c79c',
 8: '68230896-6d12-43f2-9381-fe0a7dd30338',
 9: '9e6994c6-6885-40a6-8def-dca58d14252f',
 10: 'd72c1965-1a53-4486-af7f-4dfb89e840b2',
 11: 'b27f02bc-35f2-4dd3-a03d-124de2fe914a',
 12: 'bf2f2189-df71-41db-bdc6-42277119c9d2',
 13: '0cd46655-5367-400d-a202-f72f2891bb36',
 14: '9731a7da-9560-4405-9daf-97b76a0a76b3',
 15: '22b09ce5-d02a-4287-918b-a3b5e123f5e0',
 16: '7ca23d38-3d4f-405f-b380-25ae398cd8fc',
 17: 'ee824207-1f18-4e12-9ac1-fe81b8626acd',
 18: '2eea0e88-cbba-4007-9697-62b15dbd594f',
 19: '7d6e607d-50fc-4f5e-8968-3846c17dcce9',
 20: 'aa7c489c-b418-46eb-a2c5-000d7618f2fe',
 21: '408e5c7f-d434-4c4a-b9af-d763065c9543',
 22: 'cee0fc76-f33f-

In [42]:
vector_store.get_by_ids(['f5284155-6e85-420c-bc61-aa5709ab2cbc'])

[]

In [43]:
retriver= vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})
retriver


VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x714776285250>, search_kwargs={'k': 4})

In [44]:
retriver.invoke('what is the whole video about give me sumary')

[Document(id='c65e10c6-c8e3-4dd6-81c7-bb75710ca5f8', metadata={}, page_content="[1720.8s] more natural and gives your brain one or\n[1723.2s] two extra seconds. For For\n[1725.8s] let me think. I usually practice after\n[1728.4s] dinner. Let me think. I saw him last\n[1731.4s] Friday. Let me think. Yes, I think that\n[1734.9s] is a good idea. I can also say, let me\n[1737.8s] think for a second. Very good. A useful\n[1740.7s] note, let me think is not only when you\n[1743.4s] are confused, it is also a natural\n[1745.7s] conversation tool.\n[1747.2s] >> My personal connection is, let me think\n[1749.7s] saves me every day.\n[1751.5s] >> Great. Now, let's do a quick summary.\n[1754.2s] Translate. Change words from one\n[1756.8s] language to another.\n[1758.4s] >> Habit. Something you do regularly.\n[1760.8s] >> Describe. Say what something is like.\n[1763.7s] >> Notice. Become aware of something.\n[1766.1s] >> Repeat. Say or do again.\n[1768.8s] >> Shadowing. Listening and repeating rig

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0.2)

In [45]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", temperature=0.2)

In [46]:
prompt=PromptTemplate(
    template="""
    You are ahelpful assisstant .
    Answer only from the provided transcript context .
    if the context is insuffoecient ,just say you dont know .
    {context}
    Question:{question}
    """,
    input_variables=['context','question'])

In [47]:
question = "is topic of fighting is disscussed in the video,if yes where?"
retrieved_docs = retriver.invoke(question)


In [48]:
retrieved_docs

[Document(id='d0bd7f1d-7576-4656-8257-716092c30408', metadata={}, page_content="[48.7s] >> We'll talk about daily habits, simple\n[50.6s] practice, common mistakes, and useful\n[53.0s] phrases you can start using right away.\n[55.6s] >> If you want more easy English lessons\n[57.8s] like this, please like this video and\n[60.2s] subscribe to our channel.\n[61.6s] >> And click the bell, too. Your future\n[63.5s] English-speaking brain will thank you.\n[66.0s] >> All right.\n[66.9s] >> Let's get started.\n[69.8s] >> Leo, how are you today?\n[71.8s] >> I'm good, but a little sleepy. I stayed\n[74.3s] up late last night.\n[75.5s] >> Oh, no. Why did you stay up late?\n[77.8s] >> I told myself I will sleep at 10:00.\n[81.2s] Then I watched one short video, then\n[83.0s] another, then another. Suddenly, it was\n[85.6s] midnight.\n[86.6s] >> That is a very modern problem.\n[89.3s] >> Yes. My plan was strong, but my phone\n[92.2s] was stronger.\n[93.4s] >> I'm doing well today. I had a quiet\n[

In [49]:
context_text="\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"[48.7s] >> We'll talk about daily habits, simple\n[50.6s] practice, common mistakes, and useful\n[53.0s] phrases you can start using right away.\n[55.6s] >> If you want more easy English lessons\n[57.8s] like this, please like this video and\n[60.2s] subscribe to our channel.\n[61.6s] >> And click the bell, too. Your future\n[63.5s] English-speaking brain will thank you.\n[66.0s] >> All right.\n[66.9s] >> Let's get started.\n[69.8s] >> Leo, how are you today?\n[71.8s] >> I'm good, but a little sleepy. I stayed\n[74.3s] up late last night.\n[75.5s] >> Oh, no. Why did you stay up late?\n[77.8s] >> I told myself I will sleep at 10:00.\n[81.2s] Then I watched one short video, then\n[83.0s] another, then another. Suddenly, it was\n[85.6s] midnight.\n[86.6s] >> That is a very modern problem.\n[89.3s] >> Yes. My plan was strong, but my phone\n[92.2s] was stronger.\n[93.4s] >> I'm doing well today. I had a quiet\n[95.4s] morning, so I feel calm and ready.\n[97.9s] >> That sounds very Tina. Qu

In [50]:
final_prompt = prompt.invoke({"context": context_text, "question": question})
final_prompt

StringPromptValue(text="\n    You are ahelpful assisstant .\n    Answer only from the provided transcript context .\n    if the context is insuffoecient ,just say you dont know .\n    [48.7s] >> We'll talk about daily habits, simple\n[50.6s] practice, common mistakes, and useful\n[53.0s] phrases you can start using right away.\n[55.6s] >> If you want more easy English lessons\n[57.8s] like this, please like this video and\n[60.2s] subscribe to our channel.\n[61.6s] >> And click the bell, too. Your future\n[63.5s] English-speaking brain will thank you.\n[66.0s] >> All right.\n[66.9s] >> Let's get started.\n[69.8s] >> Leo, how are you today?\n[71.8s] >> I'm good, but a little sleepy. I stayed\n[74.3s] up late last night.\n[75.5s] >> Oh, no. Why did you stay up late?\n[77.8s] >> I told myself I will sleep at 10:00.\n[81.2s] Then I watched one short video, then\n[83.0s] another, then another. Suddenly, it was\n[85.6s] midnight.\n[86.6s] >> That is a very modern problem.\n[89.3s] >> Yes. My

In [51]:
answer=llm.invoke(final_prompt)

print(answer.content)

/home/muditas/Documents/RAG/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:4059: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': "I don't know.", 'extras': {'signature': 'EpYBCpMBAWkUfRMPJ6BDGjdl3YrS3KKkiPDQJYhn5R4jLL58+78FD/HmIc402dvExsy4+/YMLVgZSRaxRIIRoDtCtEpc72/ReVUuGrelAk+7Zkdw0S7M20/rQCkGPBPpxa1uLXyd7vn60BG6bwqn/ln/2wIOapMOvmXraEVdqVt3V+UHhibriSM3Bzvqb7aTJsfHWMKAOC2E'}}]


In [52]:
target_ranges = [(445, 460), (475, 485), (1010, 1025)]

for line in transcript.split("\n"):
    if line.startswith("["):
        # Extract the number inside [123.4s]
        timestamp = float(line.split("s]")[0].replace("[", ""))
        
        if any(start <= timestamp <= end for start, end in target_ranges):
            print(line)

[445.8s] >> I talk to myself all the time.
[447.8s] >> Really?
[448.7s] >> Yes, mostly about food. Leo, do you need
[451.8s] another cookie? Leo, probably not. Leo,
[455.4s] maybe yes.
[456.7s] >> That is not exactly the study method I
[459.1s] planned, but it is still self-talk.
[475.2s] wash my face. I make breakfast. I go to
[478.1s] work at 8:30. At lunch, I eat rice and
[480.9s] vegetables. In the evening, I watch a
[483.2s] video and read for 20 minutes.
[1010.4s] make breakfast.
[1011.7s] >> Perfect. What would you say?
[1013.8s] >> I'm hungry. I need something fast. I'm
[1016.6s] cutting bread. I'm frying an egg. Oops,
[1019.8s] too much salt.
[1021.2s] >> Very realistic.
[1022.9s] >> Too realistic.
[1024.3s] >> My tiny English moment is often when I


In [53]:
from langchain_core.runnables import RunnableParallel,RunnableLambda,RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [54]:
def format_docs(retrived_docs):
    context_text="\n\n".join(doc.page_content for doc in retrived_docs)
    return context_text

In [55]:
parallel_chain=RunnableParallel({
    'context':retriver|RunnableLambda(format_docs),
    'question':RunnablePassthrough()
})

In [56]:
parallel_chain.invoke('what are the food name used in video')

{'context': '[999.6s] >> Oh, I like that. It feels light, not\n[1002.3s] heavy.\n[1003.2s] >> Yes, you\'re not changing your whole\n[1005.3s] life. You\'re building one habit.\n[1008.3s] >> My tiny English moment could be when I\n[1010.4s] make breakfast.\n[1011.7s] >> Perfect. What would you say?\n[1013.8s] >> I\'m hungry. I need something fast. I\'m\n[1016.6s] cutting bread. I\'m frying an egg. Oops,\n[1019.8s] too much salt.\n[1021.2s] >> Very realistic.\n[1022.9s] >> Too realistic.\n[1024.3s] >> My tiny English moment is often when I\n[1026.8s] prepare my to-do list. I say, "First, I\n[1030.2s] answer emails. Then, I record a lesson.\n[1033.5s] After that, I review vocabulary."\n[1036.3s] >> That sounds very organized.\n[1038.2s] >> It is.\n[1039.4s] >> Another thing that helps me is changing\n[1041.2s] my phone language to English.\n[1043.4s] >> Yes, that can help if your level is\n[1045.3s] comfortable enough.\n[1046.9s] >> At first, I got confused. I looked for\n[1049.3s] settin

In [59]:
parser=StrOutputParser()
main_chain=parallel_chain|prompt|llm|parser


In [60]:
main_chain.invoke('summarize the whole video')

/home/muditas/Documents/RAG/.venv/lib/python3.12/site-packages/langchain_google_genai/chat_models.py:4059: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'Based on the provided transcript, here is a summary of what was discussed:\n\n* **Reducing Translation:** Learners often translate too much in their heads. The goal is to connect English words directly to ideas, pictures, and daily actions using phrases like "think in English" and "don\'t translate every word."\n* **Daily Practice:** Progress comes from small habits and daily practice (5 or 10 minutes a day). Learners can name objects, describe actions, and talk about plans (e.g., "This is my desk," "I\'m making breakfast").\n* **Building Longer Answers:** Answers can be built step by step by starting with a short sentence and adding a reason or detail, using helpful phrases like "I usually... because..."\n* **Mistakes and Confidence:** Mistakes are normal and useful, whereas silence is often a bigger problem. Useful phrases to buy time and maintain conversation include "Let me think" and "I\'m not sure."\n* **Tools and Methods:** Helpful methods include listening and repeating, shado